# 📉 Recession Prediction: Timeseries Data Collection & EDA

**Goal:** Collect macroeconomic timeseries data (2004–present) and correlate leading/lagging indicators with the Hamilton GDP-Based Recession Probability Index.

**Data Sources:**
- 🏦 **FRED** (Federal Reserve Economic Data) — GDP, CPI, Interest Rates, PCE, Unemployment, Hamilton Index
- 🔍 **Google Trends** — recession-related search queries as a sentiment proxy

**Target Variable:** `JHGDPBRINDX` — Hamilton GDP-Based Recession Index (continuous 0–100 probability scale, replacing binary NBER 0/1 flag)

**Notebook Structure:**
1. Setup & API Key
2. FRED Data Collection
3. Google Trends Data Collection
4. Data Cleaning & Alignment
5. Exploratory Visualization
6. Lagged Correlation Analysis
7. OLS Regression — Feature vs Hamilton Index

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from fredapi import Fred
from pytrends.request import TrendReq
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import time

# --- Plot style ---
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

RECESSION_COLOR = '#ff4444'
START_DATE = '1989-01-01'
END_DATE   = '2025-12-31'

print('✅ Libraries loaded')

✅ Libraries loaded


In [2]:
# ============================================================
#  🔑  PASTE YOUR FREE FRED API KEY HERE
#  Get one at: https://fred.stlouisfed.org/docs/api/api_key.html
# ============================================================
FRED_API_KEY = 'b03cb386e1eb46026485472220367b78'

fred = Fred(api_key=FRED_API_KEY)
print('✅ FRED client initialized')

✅ FRED client initialized


In [ ]:
# from pytrends.request import TrendReq
# pytrends = TrendReq(hl='en-US', tz=360)
# print('✅ pytrends initialized')

✅ pytrends initialized


## 2. FRED Data Collection

| Series ID | Description | Frequency |
|-----------|-------------|----------|
| `GDPC1`        | Real GDP (Chained 2017 $) | Quarterly |
| `CPIAUCSL`     | CPI – All Urban Consumers | Monthly |
| `FEDFUNDS`     | Federal Funds Effective Rate | Monthly |
| `T10Y2Y`       | 10Y–2Y Treasury Yield Spread | Daily → Monthly |
| `UNRATE`       | Unemployment Rate | Monthly |
| `PCECC96`      | Real Personal Consumption Expenditures | Monthly |
| `USREC`        | NBER Recession Indicator (0/1) — used for chart shading only |
| `JHGDPBRINDX`  | **Hamilton GDP-Based Recession Probability — TARGET VARIABLE** | Quarterly |

In [3]:
def fetch_fred(series_id, name, start=START_DATE, end=END_DATE, freq=None):
    """Fetch a FRED series, optionally resample to monthly."""
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s.name = name
    if freq:
        s = s.resample(freq).mean()
    print(f'  ✅ {name:45s}  {len(s):>5} obs  [{s.index.min().date()} → {s.index.max().date()}]')
    return s

print('Fetching FRED series...')
gdp          = fetch_fred('GDPC1',       'Real GDP (Billions $)',              freq='QS')
cpi          = fetch_fred('CPIAUCSL',    'CPI (All Urban)',                    freq='MS')
fed_funds    = fetch_fred('FEDFUNDS',    'Fed Funds Rate (%)',                 freq='MS')
yield_spread = fetch_fred('T10Y2Y',      '10Y-2Y Yield Spread (%)',            freq='MS')
unemployment = fetch_fred('UNRATE',      'Unemployment Rate (%)',              freq='MS')
pce          = fetch_fred('PCECC96',     'Real PCE (Consumer Spending)',       freq='MS')
recession    = fetch_fred('USREC',       'NBER Recession Indicator (0/1)',     freq='MS')
hamilton     = fetch_fred('SAHMREALTIME','Hamilton Recession Probability (%)', freq='MS')

print('\n✅ All FRED series fetched!')

Fetching FRED series...
  ✅ Real GDP (Billions $)                            148 obs  [1989-01-01 → 2025-10-01]
  ✅ CPI (All Urban)                                  444 obs  [1989-01-01 → 2025-12-01]
  ✅ Fed Funds Rate (%)                               444 obs  [1989-01-01 → 2025-12-01]
  ✅ 10Y-2Y Yield Spread (%)                          444 obs  [1989-01-01 → 2025-12-01]
  ✅ Unemployment Rate (%)                            444 obs  [1989-01-01 → 2025-12-01]
  ✅ Real PCE (Consumer Spending)                     442 obs  [1989-01-01 → 2025-10-01]
  ✅ NBER Recession Indicator (0/1)                   444 obs  [1989-01-01 → 2025-12-01]
  ✅ Hamilton Recession Probability (%)               444 obs  [1989-01-01 → 2025-12-01]

✅ All FRED series fetched!


In [4]:
# --- Compute derived features ---

# YoY CPI inflation rate
inflation = cpi.pct_change(12) * 100
inflation.name = 'YoY Inflation Rate (%)'

# QoQ Real GDP growth (annualized)
gdp_growth = gdp.pct_change(1) * 400
gdp_growth.name = 'Real GDP Growth (%, annualized)'

# YoY PCE growth
pce_growth = pce.pct_change(12) * 100
pce_growth.name = 'YoY PCE Growth (%)'

print('Derived features computed:')
print(f'  Inflation series:  {len(inflation)} months')
print(f'  GDP growth series: {len(gdp_growth)} quarters')
print(f'  PCE growth series: {len(pce_growth)} months')

Derived features computed:
  Inflation series:  444 months
  GDP growth series: 148 quarters
  PCE growth series: 442 months


### About JHGDPBRINDX — Hamilton GDP-Based Recession Index

Developed by James Hamilton (UC San Diego), this series estimates the **probability that the U.S. economy is in recession** each quarter, using a Markov-switching model fitted to real GDP growth. Values range from 0 (no recession signal) to 100 (near-certain recession). Unlike the NBER binary flag, which is determined retrospectively and announced with a long lag, the Hamilton index is model-based and provides a **continuous, graded signal** — making it a richer regression target.

We keep `USREC` only for shading recession bands on charts. The Hamilton index replaces it as the **target variable** for all correlation and regression analysis.

In [5]:
# Quick look at the Hamilton index
hamilton.describe()

count    443.000000
mean       0.397630
std        1.078433
min       -0.370000
25%       -0.030000
50%        0.030000
75%        0.300000
max        9.500000
Name: Hamilton Recession Probability (%), dtype: float64

## 3. Google Trends Data

Monthly search interest for recession-related keywords as a public sentiment proxy (0–100 relative index).

In [14]:
def fetch_trends(keywords, timeframe='all', geo='US', sleep=20, retries=4, backoff=2):
    """
    Fetch Google Trends for up to 5 keywords at once.
    Uses a browser User-Agent and longer delays to reduce 429 rate-limiting.
    Returns a DataFrame with monthly interest (0-100 index).
    """
    requests_args = {
        'headers': {
            'User-Agent': (
                'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/120.0.0.0 Safari/537.36'
            )
        }
    }
    pytrends = TrendReq(hl='en-US', tz=360, timeout=(10, 25), requests_args=requests_args)
    for attempt in range(1, retries + 1):
        try:
            pytrends.build_payload(keywords, cat=0, timeframe=timeframe, geo=geo)
            time.sleep(sleep)
            df = pytrends.interest_over_time()
            if 'isPartial' in df.columns:
                df = df.drop(columns=['isPartial'])
            print(f'  ✅ Fetched trends for: {keywords}  → {len(df)} data points')
            return df
        except Exception as e:
            if attempt == retries:
                print(f'All retries exhausted for {keywords}: {e}')
                raise
            wait = sleep * (backoff ** attempt)
            print(f'Attempt {attempt} failed ({e}), retrying in {wait}s...')
            time.sleep(wait)

print('Fetching Google Trends (this may take ~30 seconds)...')
trends_recession = fetch_trends(['recession', 'unemployment', 'layoffs'])
time.sleep(3)
trends_economy   = fetch_trends(['economic crisis', 'inflation'])

trends = pd.concat([trends_recession, trends_economy], axis=1)
trends.index = trends.index.to_period('M').to_timestamp('M')
trends = trends.loc[START_DATE:END_DATE]
trends.columns = ['trend_recession', 'trend_unemployment', 'trend_layoffs', 'trend_economic_crisis', 'trend_inflation']

print(f'\n✅ Google Trends shape: {trends.shape}')
trends.head()

Fetching Google Trends (this may take ~30 seconds)...


KeyboardInterrupt: 

In [ ]:
# trends.to_csv('google_trends.csv')
# print('✅ Saved google_trends.csv')

## 4. Data Cleaning & Alignment

All series are resampled to **monthly frequency** and joined into a single master DataFrame. Quarterly series (GDP, Hamilton) are forward-filled within each quarter.

In [10]:
# Align to monthly index starting from 2004 (when Google Trends begins)
monthly_idx = pd.date_range(start=START_DATE, end=END_DATE, freq='M')
def align(series, idx=monthly_idx, method='ffill'):
    """Reindex series to monthly, forward-fill gaps."""
    s = series.copy()
    s.index = s.index.to_period('M').to_timestamp('M')
    return s.reindex(idx).fillna(method=method) if method else s.reindex(idx)

def align_interpolate(series, idx=monthly_idx):
    """Reindex quarterly series to monthly, interpolate linearly between observations."""
    s = series.copy()
    s.index = s.index.to_period('M').to_timestamp('M')
    return s.reindex(idx).interpolate(method='time', limit_direction='forward')

master = pd.DataFrame(index=monthly_idx)

# Monthly series — ffill is fine, these are already monthly
master['cpi']           = align(cpi)
master['inflation']     = align(inflation)
master['fed_funds']     = align(fed_funds)
master['yield_spread']  = align(yield_spread)
master['unemployment']  = align(unemployment)
master['pce_growth']    = align(pce_growth)
master['hamilton']      = align_interpolate(hamilton)

# Quarterly series — interpolate instead of forward-fill
master['gdp_growth']    = align_interpolate(gdp_growth)
# master['hamilton']      = align_interpolate(hamilton)

# Recession stays ffilled (binary 0/1 NBER indicator, carry-forward is correct)
master['recession']     = align(recession).round().astype('Int64')

In [11]:
master.dropna(inplace=True)  # drop pre-1990 rows where Hamilton is NaN

In [12]:
master.shape

(432, 9)

In [13]:
master.head(24)

,cpi,inflation,fed_funds,yield_spread,unemployment,pce_growth,hamilton,gdp_growth,recession
1990-01-31,127.5,5.198020,8.23,0.121429,5.4,2.737589,0.13,4.371528,0
1990-02-28,128.0,5.263158,8.24,0.102632,5.3,2.737589,0.13,3.452987,0
1990-03-31,128.6,5.237316,8.28,-0.038182,5.2,2.737589,0.10,2.436031,0
1990-04-30,128.9,4.711617,8.26,0.061500,5.4,2.571849,0.13,1.451880,0
1990-05-31,129.1,4.365400,8.18,0.115909,5.4,2.571849,0.13,1.052400,0
1990-06-30,129.9,4.673650,8.29,0.129048,5.2,2.571849,0.03,0.665807,0
1990-07-31,130.5,4.819277,8.15,0.314286,5.5,1.990863,0.07,0.266327,0
1990-08-31,131.6,5.702811,8.13,0.691304,5.7,1.990863,0.17,-1.050481,1
1990-09-30,132.5,6.169872,8.20,0.813684,5.9,1.990863,0.33,-2.324811,1
1990-10-31,133.4,6.379585,8.11,0.841818,5.9,0.776979,0.40,-3.641618,1


In [15]:
master.to_csv('macro_data_interpolated_Sahm.csv')

In [18]:
trends = pd.read_csv('/Users/pranavmoses/Desktop/ADTA5640/Project/google_trends.csv', index_col=0, parse_dates=True)
trends.tail()

,trend_recession,trend_unemployment,trend_layoffs,trend_economic crisis,trend_inflation
date,,,,,
2025-08-31,0,5,1,1,50
2025-09-30,1,5,1,1,48
2025-10-31,1,5,2,1,53
2025-11-30,1,5,1,2,67
2025-12-31,1,6,1,2,71


In [19]:
trends.index = pd.to_datetime(trends.index).to_period('M').to_timestamp('M')

# Left join — keeps all master rows; months before 2004 will have NaN for trend columns
master = master.join(trends, how='left')

# Two versions of master:
# master_full   → all months from START_DATE, NaN in trend columns before 2004
# master_trends → only months where Google Trends data exists (2004 onwards)
master_full   = master.copy()
master_trends = master.dropna(subset=list(trends.columns))
master_trends.shape, master_full.shape, master_full.head()

((264, 14),
 (432, 14),
               cpi  inflation  fed_funds  yield_spread  unemployment  \
 1990-01-31  127.5   5.198020       8.23      0.121429           5.4   
 1990-02-28  128.0   5.263158       8.24      0.102632           5.3   
 1990-03-31  128.6   5.237316       8.28     -0.038182           5.2   
 1990-04-30  128.9   4.711617       8.26      0.061500           5.4   
 1990-05-31  129.1   4.365400       8.18      0.115909           5.4   
 
             pce_growth  hamilton  gdp_growth  recession  trend_recession  \
 1990-01-31    2.737589      0.13    4.371528          0              NaN   
 1990-02-28    2.737589      0.13    3.452987          0              NaN   
 1990-03-31    2.737589      0.10    2.436031          0              NaN   
 1990-04-30    2.571849      0.13    1.451880          0              NaN   
 1990-05-31    2.571849      0.13    1.052400          0              NaN   
 
             trend_unemployment  trend_layoffs  trend_economic crisis  \
 1990

In [20]:
master_trends.to_csv('macro_data_interpolated_Sahm_with_trends.csv')